# OceanEye — video analysis notebook (perception + risk preview)

Runs the **same code as the prototype worker**: YOLO person detection → ByteTrack → danger zone → behaviour rules →
risk score → candidate events with evidence (thumbnail + H.264 clip). No API/server needed.

**Kernels** – detected automatically:
* **Local `venv` kernel (Antigravity / VS Code):** uses `C:\pbl_oceaneye\app` and `C:\pbl_oceaneye\videos` directly.
* **Colab runtime (Colab extension, GPU):** uses the code embedded in this notebook (2026-10-08 05:21 UTC) + an upload widget.

**Crash-safe by design:** CPU threads are capped, the CPU profile uses a small model / input size / frame rate and a
frame limit, one video at a time, and videos are not embedded in the notebook unless you ask. Increase limits only on a GPU.
Rebuild after code changes: `python scripts\build_colab_notebook.py`.

## 1 · Hardware check

In [ ]:
import importlib.util, os, platform, subprocess, sys
IN_COLAB = importlib.util.find_spec("google.colab") is not None
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=20).stdout.strip()
except (FileNotFoundError, subprocess.TimeoutExpired):
    gpu = ""
print("python", sys.version.split()[0], "|", platform.platform(), "| Colab runtime:", IN_COLAB)
print("NVIDIA GPU:", gpu or "none")

## 2 · Load the OceanEye code
`auto` = your project folder when this kernel can see it, otherwise the embedded copy (first cell below – one long line, just run it).

In [ ]:
# Embedded OceanEye code (app/ + pyproject.toml), built 2026-10-08 05:21 UTC. Generated by scripts/build_colab_notebook.py - do not edit.
EMBEDDED_BUNDLE = "..."

In [ ]:
CODE_SOURCE = "auto"   # auto | local | embedded
import base64, io, shutil, zipfile
from pathlib import Path

def _find_project(start: Path):
    for p in [start, *start.parents, Path(r"C:\pbl_oceaneye")]:
        if (p / "app").is_dir() and (p / "pyproject.toml").is_file():
            return p
    return None

if CODE_SOURCE == "auto":
    CODE_SOURCE = "local" if _find_project(Path.cwd()) else "embedded"
if CODE_SOURCE == "local":
    ROOT = _find_project(Path.cwd())
else:
    ROOT = Path("/content/oceaneye") if IN_COLAB else Path.home() / "oceaneye_runtime"
    shutil.rmtree(ROOT / "app", ignore_errors=True)
    ROOT.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(io.BytesIO(base64.b64decode(EMBEDDED_BUNDLE))).extractall(ROOT)
for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[name]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("code:", CODE_SOURCE, "| root:", ROOT)

## 3 · Dependencies (installs only what is missing)

In [ ]:
need = {"ultralytics": "ultralytics", "lap": "lap", "cv2": "opencv-python", "imageio_ffmpeg": "imageio-ffmpeg",
        "pandas": "pandas", "matplotlib": "matplotlib"}
if CODE_SOURCE != "local":
    need["ipywidgets"] = "ipywidgets"
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import cv2, torch, ultralytics
CUDA = torch.cuda.is_available()
print(f"torch {torch.__version__} | CUDA usable: {CUDA} | ultralytics {ultralytics.__version__} | opencv {cv2.__version__}")
if gpu and not CUDA:
    print("\n⚠ An NVIDIA GPU is present but this PyTorch build is CPU-only ('+cpu').\n"
          "  Install the CUDA build in your venv (see docs/prototype.md -> GPU) to make analysis ~5-10x faster.")

## 4 · Resource limits + automatic profile

The profile comes from `app.vision.profiles` - the **same policy the prototype's worker uses**, so what
you measure here is what the live system will do on comparable hardware. It sizes the model, input size
and frame rate to the machine, which is what keeps a small GPU from running out of memory.

In [ ]:
from app.shared.hardware import probe as probe_hardware
from app.vision.profiles import choose_profile

CORES = os.cpu_count() or 4
THREADS = max(1, min(4, CORES // 2))
torch.set_num_threads(THREADS); cv2.setNumThreads(1)

HW = probe_hardware()
DEVICE = "cuda:0" if CUDA else "cpu"
AUTO = choose_profile(DEVICE, vram_gb=HW.vram_gb, ram_gb=HW.ram_gb, cpu_count=HW.cpu_count, workers_sharing=1)
PROFILE = dict(MODEL=AUTO.model, IMGSZ=AUTO.imgsz, DEVICE=DEVICE, TARGET_FPS=AUTO.target_fps,
               # On the CPU a frame cap keeps a first run to a couple of minutes; on a GPU, run it all.
               MAX_FRAMES=None if CUDA else 450)
print(f"machine: {HW.describe()} | threads={THREADS}/{CORES}")
print(f"profile: {AUTO.describe()}")
print(PROFILE)

## 5 · Videos
Local: everything in `videos\` is listed. Colab runtime: use the upload widget, then run the next cell.

In [ ]:
VIDEO_DIR = ROOT / "videos" if CODE_SOURCE == "local" else (Path("/content/videos") if IN_COLAB else ROOT / "videos")
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
EXT = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}
uploader = None
if CODE_SOURCE != "local":
    import ipywidgets as widgets
    from IPython.display import display
    uploader = widgets.FileUpload(accept=",".join(sorted(EXT)), multiple=True, description="Upload videos")
    display(uploader)

In [ ]:
def _items(up):
    if up is None: return []
    v = up.value
    return [(n, i["content"]) for n, i in v.items()] if isinstance(v, dict) else [(i["name"], i["content"]) for i in v]
for name, content in _items(uploader):
    (VIDEO_DIR / Path(name).name).write_bytes(bytes(content))
VIDEOS = sorted(p for p in VIDEO_DIR.iterdir() if p.suffix.lower() in EXT)
for i, p in enumerate(VIDEOS):
    cap = cv2.VideoCapture(str(p)); fps = cap.get(cv2.CAP_PROP_FPS) or 0; n = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    print(f"[{i}] {p.name}  {int(cap.get(3))}x{int(cap.get(4))}  {n / fps if fps else 0:.0f}s @ {fps:.0f} fps"); cap.release()
assert VIDEOS, f"no videos in {VIDEO_DIR}"

## 6 · Settings
* `VIDEO_INDEX` – which video from step 5. One at a time on CPU.
* `ZONE` – danger-zone polygon in normalised coordinates (0..1, origin top-left). `None` = whole frame.
  The default (lower 60 % of the image) is only a demo guess – draw real zones in the Control Room.
* `SENSITIVITY` – low / medium / high (experimental presets; high = more alerts and more false alarms).

In [ ]:
VIDEO_INDEX = 0
ZONE = [[0, 0.4], [1, 0.4], [1, 1], [0, 1]]
SENSITIVITY = "medium"
MODEL, IMGSZ, DEVICE = PROFILE["MODEL"], PROFILE["IMGSZ"], PROFILE["DEVICE"]
TARGET_FPS, MAX_FRAMES = PROFILE["TARGET_FPS"], PROFILE["MAX_FRAMES"]
VIDEO = VIDEOS[VIDEO_INDEX]
print(f"{VIDEO.name} | {MODEL}@{IMGSZ} on {DEVICE} | {TARGET_FPS} fps analysed | max frames {MAX_FRAMES} | {SENSITIVITY}")

## 7 · Run (detection → tracking → zone → behaviour → risk → events)

In [ ]:
import json, time
from datetime import datetime
from app.worker.offline import analyze_video
OUT = ROOT / "outputs" / "notebook" / f"{VIDEO.stem}_{datetime.now():%Y%m%d_%H%M%S}"
t0 = time.time()
RESULT = analyze_video(str(VIDEO), str(OUT), zone_polygon=ZONE, model=MODEL, imgsz=IMGSZ, device=DEVICE,
                       target_fps=TARGET_FPS, sensitivity=SENSITIVITY, max_frames=MAX_FRAMES,
                       weights_dir=str(ROOT / "models"), camera_name=VIDEO.stem[:30])
(OUT / "events.json").write_text(json.dumps(RESULT["events"], indent=2), encoding="utf-8")
print(f"done in {time.time() - t0:.0f}s ->", OUT)
print(json.dumps(RESULT["summary"], indent=2))

## 8 · Results: annotated frames + candidate events

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
from IPython.display import display
cap = cv2.VideoCapture(RESULT["overlay_video"]); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
fig, axes = plt.subplots(2, 3, figsize=(15, 6)); fig.suptitle(VIDEO.name)
for ax, k in zip(axes.flat, range(6)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(k * (n - 1) / 5)); ok, img = cap.read()
    if ok: ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.axis("off")
cap.release(); plt.tight_layout(); plt.show()

ev = RESULT["events"]
print(f"{len(ev)} candidate events (heuristic – review each one; they are NOT confirmed drownings)")
if ev:
    display(pd.DataFrame([{"time": e["started_at"][11:19], "track": e["track_id"], "risk": e["risk_score"], "level": e["risk_level"],
                           "signals": ", ".join(e["distress_signals"]), "reason": e["reason"][:120]} for e in ev]))
    thumbs = [Path(RESULT["evidence_dir"]) / e["evidence"]["thumbnail_key"] for e in ev[:6]]
    fig, axes = plt.subplots(1, len(thumbs), figsize=(5 * len(thumbs), 3.5), squeeze=False)
    for ax, t in zip(axes[0], thumbs):
        ax.imshow(cv2.cvtColor(cv2.imread(str(t)), cv2.COLOR_BGR2RGB)); ax.axis("off")
    plt.show()
PLAY_VIDEO = False   # embedding a video makes the notebook heavy; open overlay.mp4 from the output folder instead
if PLAY_VIDEO and Path(RESULT["overlay_video"]).stat().st_size < 25e6:
    from IPython.display import Video
    display(Video(RESULT["overlay_video"], embed=True, width=720))
print("overlay video:", RESULT["overlay_video"])

## 9 · Tracking & risk over time
Many more unique IDs than people at once = fragmented tracks; risk spikes show where the rules fired.

In [ ]:
rows = []
for line in open(RESULT["tracks_jsonl"], encoding="utf-8"):
    r = json.loads(line)
    for t in r["tracks"]:
        x1, y1, x2, y2 = t["bbox"]
        rows.append(dict(t=r["video_time_s"], id=t["track_id"].rsplit("-", 1)[-1], cx=(x1 + x2) / 2, cy=(y1 + y2) / 2, risk=t.get("risk", 0)))
df = pd.DataFrame(rows)
if df.empty:
    print("no people tracked – try IMGSZ 960/1280 or a bigger model (GPU)")
else:
    print(f"unique IDs {df.id.nunique()} | max people at once {df.groupby('t').id.nunique().max()}")
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
    df.groupby("t").id.nunique().plot(ax=ax[0], title="people tracked")
    df.groupby("t").risk.max().plot(ax=ax[1], title="max risk score", ylim=(0, 1))
    for _, g in df.groupby("id"): ax[2].plot(g.cx, g.cy, lw=1)
    ax[2].invert_yaxis(); ax[2].set_title("trajectories"); plt.tight_layout(); plt.show()

## 10 · (Colab runtime only) download results
Local kernel: results are already in `outputs\notebook\`.

In [ ]:
if CODE_SOURCE != "local":
    from IPython.display import HTML
    z = shutil.make_archive(str(OUT), "zip", OUT)
    data = base64.b64encode(Path(z).read_bytes()).decode()
    display(HTML(f'<a download="{Path(z).name}" href="data:application/zip;base64,{data}">⬇ Download {Path(z).name} ({Path(z).stat().st_size / 1e6:.1f} MB)</a>'))
else:
    print("saved in", OUT)

## Next: the full prototype (dashboards + live alerts)
```powershell
cd C:\pbl_oceaneye
.\venv\Scripts\Activate.ps1
python scripts\run_api.py            # then open http://127.0.0.1:8000
```
Control Room · Lifeguard · Video Analysis – see `docs/prototype.md`.